In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import json
from pathlib import Path
import sys
from torchvision.transforms import transforms
from functools import partial

In [ ]:
ROOT_DIR = Path.cwd().parent
ROOT_DIR

In [ ]:
sys.path.extend([str(ROOT_DIR)])

In [ ]:
from model.ImageCaptionModel import *
from src.preprocessing_data.ComputeMeanStd import *
from src.custom_dataset.organize_data import *
from src.NLP.build_vocab import *

In [ ]:
computer = Compute_mean_std()

In [ ]:
mean, std = computer.Load_mean_std(Path(ROOT_DIR/'config'/'mean_std.json'))
mean

In [ ]:
vocab = BuildVocabFromIterator()
vocab.load_vocab(path=Path(ROOT_DIR/'config'/'vocab.json'))

In [ ]:
vocab.vocab_size

In [ ]:
dataset = ImageCaptionDataSet(
    img_dir=ROOT_DIR/'data'/'images',
    caption_dir=ROOT_DIR/'data'/'captions',
    split = 'train',
    transformation=[transforms.Normalize(mean = mean, std= std)],
    vocab=vocab
)

In [ ]:
val_dataset = dataset = ImageCaptionDataSet(
    img_dir=ROOT_DIR/'data'/'images',
    caption_dir=ROOT_DIR/'data'/'captions',
    split = 'val',
    transformation=[transforms.Normalize(mean = mean, std= std)],
    vocab=vocab
)

In [ ]:
collate = partial(collate_fn,idx_padd_token = vocab.get_padding_token())

In [ ]:
dataloader = DataLoader(dataset= dataset, batch_size=64, shuffle=True, collate_fn=collate, num_workers=2, pin_memory=True)

In [ ]:
val_dataloader = DataLoader(dataset= val_dataset, batch_size=32, collate_fn=collate, num_workers=2, pin_memory=True)

In [ ]:
model = ImageCaptionModel(
    img_size=224,
    patch_size=16,
    in_channels=3,
    embedding_dim=512,
    forward_dim=1024,
    num_heads=8,
    dropout=0.1,
    vocab_size= vocab.vocab_size,
    num_layers=3
)

In [ ]:
# Khởi tạo tất cả parameters trong Decoder bằng Kaiming (He) initialization

for p in model.transformer_decoder.parameters():
    if not isinstance(p, nn.Parameter):
        nn.init.kaiming_uniform_(p) 


In [ ]:
optimizer = torch.optim.Adam([
    {'params': model.transformer_encoder.timm_vit.parameters(),'lr': 0.0001},
    {'params': model.transformer_encoder.proj.parameters(),'lr': 0.001},
    {'params': model.transformer_decoder.parameters(), 'lr': 0.001},
    {'params': model.fc_out.parameters(), 'lr': 0.001}
], weight_decay=0.00001)

In [ ]:
# Giảm lr một lượng sau mỗi một lượng 
scheduler = torch.optim.lr_scheduler.StepLR(optimizer=optimizer, gamma=0.3, step_size=30)

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=vocab.get_padding_token())

In [ ]:
trainer = TrainModel()

In [ ]:
trainer.fit(
    model=model,
    train_loader=dataloader,
    val_loader=val_dataloader,
    n_epochs=100,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    vocab=vocab,
    num_val_batches=1
)

In [ ]:
PATH_SAVE_WEIGHT = ROOT_DIR/'model_weight.pth'

In [ ]:
torch.save(model.state_dict(), Path(PATH_SAVE_WEIGHT))